# 5. Pace & outcomes: pass execution

Compare **Direct and Indirect passes at 5v5** by preceding total possession pace and pass completion percentage.

Quartiles share the same thresholds across pass types.

Does faster pace before a pass coincide with execution?"

## 5.1. Load inputs

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROCESSED_DATA_DIR = Path("../data/processed")
PASS_TYPES = ["Direct", "Indirect"]

# Event IDs preserve order, including events sharing a clock value.
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values(
    "event_id"
)
movements = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
passes = events.loc[
    events["event"].isin(["Pass", "Incomplete Pass"])
    & events["detail_1"].isin(PASS_TYPES)
    & events["is_5v5"]
].copy()
passes["pass_type"] = passes["detail_1"]

## 5.2. Calculate preceding total possession pace

Includes movement ending at the release, but excludes the focal pass's flight to reception.

In [2]:
def add_preceding_pace(pass_events, transition_events):
    movement_groups = {
        possession_id: group
        for possession_id, group in transition_events.groupby(
            "possession_id", sort=False
        )
    }
    rows = []
    for attempt in pass_events.itertuples(index=False):
        history = movement_groups.get(attempt.possession_id, transition_events.iloc[:0])
        history = history.loc[history["end_event_id"].le(attempt.event_id)]
        duration = float(history["modeled_elapsed_seconds"].sum())
        distance = float(history["distance_ft"].sum())
        rows.append(
            {
                "event_id": attempt.event_id,
                "preceding_transitions": len(history),
                "preceding_movement_seconds": duration,
                "preceding_distance_ft": distance,
                "preceding_total_pace_ft_s": distance / duration
                if duration > 0
                else np.nan,
            }
        )
    return pass_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


passes = add_preceding_pace(passes, movements)

## 5.3. Identify pass outcomes

`Pass` means completed and `Incomplete Pass` means unsuccessful.

In [3]:
passes["complete"] = passes["event"].eq("Pass")

## 5.4. Summarize quartiles

In [4]:
def summarize_passes(data):
    pace_column = "preceding_total_pace_ft_s"
    eligible = data.loc[data[pace_column].notna()].copy()
    bands, edges = pd.qcut(
        eligible[pace_column], q=4, labels=False, retbins=True, duplicates="drop"
    )
    eligible["quartile"] = bands + 1
    boundaries = pd.DataFrame(
        {
            "quartile": np.arange(1, len(edges)),
            "lower_pace_ft_s": edges[:-1],
            "upper_pace_ft_s": edges[1:],
        }
    )
    analysis = eligible.loc[eligible["quartile"].notna()]
    summary = (
        analysis.groupby(["pass_type", "quartile"], observed=True)
        .agg(
            attempts=("event_id", "size"),
            completed=("complete", "sum"),
            games=("game_id", "nunique"),
            median_pace_ft_s=(pace_column, "median"),
        )
        .reset_index()
    )
    summary["completion_pct"] = 100 * summary["completed"] / summary["attempts"]
    summary = summary.merge(boundaries, on="quartile", validate="many_to_one")

    # Coverage counts remain visible even when preceding pace is missing.
    coverage = (
        data.assign(
            missing_pace=data[pace_column].isna(),
            charted=data["event_id"].isin(analysis["event_id"]),
        )
        .groupby("pass_type")
        .agg(
            attempts=("event_id", "size"),
            missing_pace=("missing_pace", "sum"),
            charted=("charted", "sum"),
        )
        .reset_index()
    )
    return summary, boundaries, coverage


pass_summary, quartile_boundaries, coverage_df = summarize_passes(passes)
display(coverage_df)

,pass_type,attempts,missing_pace,charted
0,Direct,11630,1096,10534
1,Indirect,6965,1352,5613


## 5.5. Display pass outcomes

Q1 is the lowest preceding total pace and Q4 the highest; hover shows numerical boundaries and counts. The two panels compare Direct and Indirect passes.

Empty groups have no percentage bar. Fewer available bands or no usable data are labeled explicitly. No uncertainty intervals are included in this first version.

In [5]:
QUARTILE_COLORS = ["#bfdbfe", "#93c5fd", "#3b82f6", "#1d4ed8"]


def pass_outcomes_figure(summary, boundaries):
    figure = make_subplots(rows=1, cols=2, subplot_titles=PASS_TYPES, shared_yaxes=True)
    band_count = len(boundaries)
    for column, pass_type in enumerate(PASS_TYPES, start=1):
        # Missing groups appear as gaps, never zero-percent results.
        bars = summary.loc[summary["pass_type"].eq(pass_type)].set_index("quartile")
        bars = bars.reindex(range(1, band_count + 1))
        figure.add_trace(
            go.Bar(
                x=[f"Q{q}" for q in bars.index],
                y=bars["completion_pct"],
                marker_color=QUARTILE_COLORS[:band_count],
                showlegend=False,
                text=[
                    f"{value:.1f}%" if pd.notna(value) else ""
                    for value in bars["completion_pct"]
                ],
                textposition="outside",
                cliponaxis=False,
                customdata=bars[
                    [
                        "attempts",
                        "completed",
                        "games",
                        "lower_pace_ft_s",
                        "upper_pace_ft_s",
                        "median_pace_ft_s",
                    ]
                ].to_numpy(),
                hovertemplate=(
                    "%{x}<br>Pass completion: %{y:.1f}%"
                    "<br>Attempts: %{customdata[0]:.0f}"
                    "<br>Completed passes: %{customdata[1]:.0f}"
                    "<br>Games: %{customdata[2]:.0f}"
                    "<br>Pace range: %{customdata[3]:.2f}"
                    " to %{customdata[4]:.2f} ft/s"
                    "<br>Median pace: %{customdata[5]:.1f} ft/s<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
    if summary.empty:
        note = "No data: no measurable pace with assigned bands."
    elif band_count < 4:
        note = f"Only {band_count} bands: ties prevent four quartiles."
    else:
        note = None
    if note:
        figure.add_annotation(
            text=note, x=0.5, y=-0.30, xref="paper", yref="paper", showarrow=False
        )
    figure.update_layout(
        title="Total pace before pass | 5v5 | pass completion",
        template="plotly_white",
        height=500,
        margin=dict(t=100, b=100),
    )
    figure.update_xaxes(
        title_text="Preceding total possession pace",
        categoryorder="array",
        categoryarray=["Q1", "Q2", "Q3", "Q4"],
    )
    figure.update_yaxes(range=[0, 100], title_text="Pass completion (%)", row=1, col=1)
    figure.update_yaxes(range=[0, 100], row=1, col=2)
    return figure


total_pace_figure = pass_outcomes_figure(pass_summary, quartile_boundaries)
total_pace_figure.show()

## 5.6. Export pass outcomes

Export the final chart summary as `pass_outcomes.parquet`: one row per pass type and quartile. `(pass_type, quartile)` identifies each row. Counts and quartiles are integers; pace and percentage fields are floating point.

The table contains pooled 5v5, total-pace pass completion results, including movement ending at the release. The future importer can insert these values directly without recalculating outcomes or quartiles.

In [6]:
# Keep the final table ordered and ready for direct database ingestion.
pass_outcomes_df = (
    pass_summary[
        [
            "pass_type",
            "quartile",
            "attempts",
            "completed",
            "games",
            "median_pace_ft_s",
            "completion_pct",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
        ]
    ]
    .astype(
        {
            "pass_type": "string",
            "quartile": "Int64",
            "attempts": "Int64",
            "completed": "Int64",
            "games": "Int64",
            "median_pace_ft_s": "Float64",
            "completion_pct": "Float64",
            "lower_pace_ft_s": "Float64",
            "upper_pace_ft_s": "Float64",
        }
    )
    .sort_values(["pass_type", "quartile"])
    .reset_index(drop=True)
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_path = PROCESSED_DATA_DIR / "pass_outcomes.parquet"
pass_outcomes_df.to_parquet(
    export_path, engine="pyarrow", compression="snappy", index=False
)

# Verify the saved values and types before reporting success.
saved = pd.read_parquet(export_path)
pd.testing.assert_frame_equal(saved, pass_outcomes_df, check_exact=True)
print(f"Exported {len(saved):,} rows and {len(saved.columns)} columns to {export_path}")
display(saved)

Exported 8 rows and 9 columns to ..\data\processed\pass_outcomes.parquet


,pass_type,quartile,attempts,completed,games,median_pace_ft_s,completion_pct,lower_pace_ft_s,upper_pace_ft_s
0,Direct,1,2362,1690,34,6.709516,71.549534,0.0,10.121004
1,Direct,2,2648,1882,34,12.980729,71.072508,10.121004,15.335979
2,Direct,3,2682,1844,34,17.649497,68.754661,15.335979,20.147318
3,Direct,4,2842,1971,34,24.129844,69.352569,20.147318,401.462327
4,Indirect,1,1675,1078,34,6.0,64.358209,0.0,10.121004
5,Indirect,2,1389,916,34,13.0,65.946724,10.121004,15.335979
6,Indirect,3,1354,881,34,17.66534,65.06647,15.335979,20.147318
7,Indirect,4,1195,796,34,23.470185,66.610879,20.147318,401.462327
